# Business Query: SQL Analysis on Gold Layer

20 business-focused SQL queries on `tpch_dev.gold` datasets covering revenue trends, customer segments, product performance, supplier analysis, geography, shipping, and returns — each paired with a business interpretation and management action.

In [0]:
%sql
-- Q01: Overall KPI Summary
SELECT
  COUNT(DISTINCT order_key)         AS total_orders,
  COUNT(*)                           AS total_line_items,
  ROUND(SUM(net_revenue), 2)         AS net_revenue,
  COUNT(DISTINCT customer_key)       AS active_customers,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT order_key), 2) AS avg_order_value
FROM tpch_dev.gold.fact_sales

**Interpretation:** This snapshot shows the full-scale business performance across all years — total orders, net revenue, active customers, and average order value in a single row.

**Management Action:** Use these KPIs as the baseline for quarterly performance reviews and executive dashboards.

In [0]:
%sql
-- Q02: Monthly Revenue Trend Across All Years
SELECT
  DATE_TRUNC('month', order_date) AS order_month,
  ROUND(SUM(net_revenue), 2)       AS monthly_revenue,
  COUNT(DISTINCT order_key)       AS monthly_orders
FROM tpch_dev.gold.fact_sales
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY order_month

**Interpretation:** Reveals seasonal patterns and month-over-month revenue momentum across all years in the dataset.

**Management Action:** Identify peak and trough months to align inventory, staffing, and marketing spend with seasonal demand cycles.

In [0]:
%sql
-- Q03: Annual Revenue with Year-over-Year Change
SELECT
  YEAR(order_date) AS sales_year,
  ROUND(SUM(net_revenue), 2) AS annual_revenue,
  LAG(ROUND(SUM(net_revenue), 2)) OVER (ORDER BY YEAR(order_date)) AS prev_year_revenue,
  ROUND(SUM(net_revenue) - LAG(SUM(net_revenue)) OVER (ORDER BY YEAR(order_date)), 2) AS yoy_change,
  ROUND((SUM(net_revenue) - LAG(SUM(net_revenue)) OVER (ORDER BY YEAR(order_date))) / LAG(SUM(net_revenue)) OVER (ORDER BY YEAR(order_date)) * 100, 2) AS yoy_pct_change
FROM tpch_dev.gold.fact_sales
GROUP BY YEAR(order_date)
ORDER BY sales_year

**Interpretation:** Shows whether the business is growing year over year and at what rate, highlighting any years with revenue contraction.

**Management Action:** Investigate any year with negative growth to understand root causes and replicate strategies from high-growth years.

In [0]:
%sql
-- Q04: Revenue by Customer Market Segment
SELECT
  dc.customer_mktsegment AS market_segment,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  COUNT(DISTINCT fs.order_key) AS total_orders
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.customer_mktsegment
ORDER BY net_revenue DESC

**Interpretation:** Identifies which customer segments generate the most revenue, showing the relative importance of each segment.

**Management Action:** Allocate marketing and account management budgets proportionally to segment revenue contribution; nurture underperforming segments with targeted campaigns.

In [0]:
%sql
-- Q05: Top 10 Customers by Net Revenue
SELECT
  dc.customer_key,
  dc.customer_name,
  dc.customer_mktsegment AS market_segment,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  COUNT(DISTINCT fs.order_key) AS total_orders
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.customer_key, dc.customer_name, dc.customer_mktsegment
ORDER BY net_revenue DESC
LIMIT 10

**Interpretation:** Reveals the most valuable individual customer relationships and their contribution to overall revenue.

**Management Action:** Assign dedicated account managers to these top customers and establish retention strategies to protect these revenue streams.

In [0]:
%sql
-- Q06: Top 10 Customers as Share of Total Revenue
WITH top_customers AS (
  SELECT
    dc.customer_key,
    dc.customer_name,
    ROUND(SUM(fs.net_revenue), 2) AS net_revenue
  FROM tpch_dev.gold.fact_sales fs
  JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
  GROUP BY dc.customer_key, dc.customer_name
  ORDER BY net_revenue DESC
  LIMIT 10
),
total_revenue AS (
  SELECT SUM(net_revenue) AS grand_total FROM tpch_dev.gold.fact_sales
)
SELECT
  tc.customer_key,
  tc.customer_name,
  tc.net_revenue,
  ROUND(tc.net_revenue / tr.grand_total * 100, 2) AS pct_of_total_revenue
FROM top_customers tc
CROSS JOIN total_revenue tr
ORDER BY tc.net_revenue DESC

**Interpretation:** Measures customer concentration risk — if the top 10 represent a large share, the business is vulnerable to losing any one of them.

**Management Action:** If top 10 share exceeds 20%, develop diversification strategies to broaden the customer base and reduce dependency risk.

In [0]:
%sql
-- Q07: Top 10 Products by Net Revenue
SELECT
  dp.product_key,
  dp.product_name,
  dp.manufacturer,
  dp.brand,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  SUM(fs.quantity) AS total_quantity
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_product dp ON fs.product_key = dp.product_key
GROUP BY dp.product_key, dp.product_name, dp.manufacturer, dp.brand
ORDER BY net_revenue DESC
LIMIT 10

**Interpretation:** Identifies the best-performing products driving the majority of revenue.

**Management Action:** Ensure supply chain reliability for these products and consider promotional emphasis to maintain their momentum.

In [0]:
%sql
-- Q08: Bottom 10 Products by Net Revenue
SELECT
  dp.product_key,
  dp.product_name,
  dp.manufacturer,
  dp.brand,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  SUM(fs.quantity) AS total_quantity
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_product dp ON fs.product_key = dp.product_key
GROUP BY dp.product_key, dp.product_name, dp.manufacturer, dp.brand
ORDER BY net_revenue ASC
LIMIT 10

**Interpretation:** Highlights products that contribute the least revenue and may be candidates for discontinuation or repositioning.

**Management Action:** Review these products for profitability and strategic value; consider phasing out underperformers to free up inventory and shelf space.

In [0]:
%sql
-- Q09: Revenue by Product Type
SELECT
  dp.product_type,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  COUNT(DISTINCT dp.product_key) AS product_count,
  SUM(fs.quantity) AS total_quantity
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_product dp ON fs.product_key = dp.product_key
GROUP BY dp.product_type
ORDER BY net_revenue DESC

**Interpretation:** Shows which product categories are the strongest revenue drivers across the catalog.

**Management Action:** Focus R&D and product development investment on high-revenue product types and evaluate whether low-revenue types need redesign or retirement.

In [0]:
%sql
-- Q10: Top 10 Suppliers by Net Revenue
SELECT
  ds.supplier_key,
  ds.supplier_name,
  ds.supplier_nation_name AS nation,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  COUNT(DISTINCT fs.product_key) AS products_supplied
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_supplier ds ON fs.supplier_key = ds.supplier_key
GROUP BY ds.supplier_key, ds.supplier_name, ds.supplier_nation_name
ORDER BY net_revenue DESC
LIMIT 10

**Interpretation:** Identifies the most critical supply chain partners by revenue contribution through their supplied products.

**Management Action:** Strengthen relationships with top suppliers through longer-term contracts and evaluate single-source risk for key products.

In [0]:
%sql
-- Q11: Revenue Share of Top 10 Suppliers
WITH top_suppliers AS (
  SELECT
    ds.supplier_key,
    ds.supplier_name,
    ROUND(SUM(fs.net_revenue), 2) AS net_revenue
  FROM tpch_dev.gold.fact_sales fs
  JOIN tpch_dev.gold.dim_supplier ds ON fs.supplier_key = ds.supplier_key
  GROUP BY ds.supplier_key, ds.supplier_name
  ORDER BY net_revenue DESC
  LIMIT 10
),
total_revenue AS (
  SELECT SUM(net_revenue) AS grand_total FROM tpch_dev.gold.fact_sales
)
SELECT
  ts.supplier_key,
  ts.supplier_name,
  ts.net_revenue,
  ROUND(ts.net_revenue / tr.grand_total * 100, 2) AS pct_of_total_revenue
FROM top_suppliers ts
CROSS JOIN total_revenue tr
ORDER BY ts.net_revenue DESC

**Interpretation:** Quantifies supplier concentration risk — a high share indicates heavy dependence on a small number of suppliers.

**Management Action:** If top 10 suppliers account for over 50% of revenue, qualify backup suppliers to mitigate supply chain disruption risk.

In [0]:
%sql
-- Q12: Revenue by Customer Region
SELECT
  fs.customer_region_name AS customer_region,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  COUNT(DISTINCT fs.customer_key) AS customer_count,
  COUNT(DISTINCT fs.order_key) AS total_orders
FROM tpch_dev.gold.fact_sales fs
GROUP BY fs.customer_region_name
ORDER BY net_revenue DESC

**Interpretation:** Shows the geographic distribution of revenue by customer region, indicating where market demand is strongest.

**Management Action:** Concentrate sales infrastructure and marketing investment in top revenue regions while exploring expansion opportunities in underperforming regions.

In [0]:
%sql
-- Q13: Revenue by Customer Nation (All Nations)
SELECT
  fs.customer_nation_name AS customer_nation,
  fs.customer_region_name AS region,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  COUNT(DISTINCT fs.customer_key) AS customer_count
FROM tpch_dev.gold.fact_sales fs
GROUP BY fs.customer_nation_name, fs.customer_region_name
ORDER BY net_revenue DESC

**Interpretation:** Provides nation-level revenue detail, revealing the top contributing countries within each region.

**Management Action:** Tailor pricing, promotions, and customer support to the purchasing patterns of each nation; prioritize high-revenue nations for account coverage.

In [0]:
%sql
-- Q14: Revenue and Volume Split by Return Flag
SELECT
  fs.return_flag,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  ROUND(SUM(fs.net_revenue) * 100 / SUM(SUM(fs.net_revenue)) OVER (), 2) AS pct_revenue,
  SUM(fs.quantity) AS total_quantity,
  COUNT(*) AS line_count
FROM tpch_dev.gold.fact_sales fs
GROUP BY fs.return_flag
ORDER BY net_revenue DESC

**Interpretation:** Measures the proportion of revenue and volume associated with returned items, indicating product quality or fulfillment issues.

**Management Action:** If returned items represent a significant share, initiate quality reviews with suppliers and investigate fulfillment processes for the affected product lines.

In [0]:
%sql
-- Q15: Average Discount by Market Segment
SELECT
  dc.customer_mktsegment AS market_segment,
  ROUND(AVG(fs.discount) * 100, 2) AS avg_discount_pct,
  ROUND(AVG(fs.discount_amount), 2) AS avg_discount_amount,
  COUNT(*) AS line_count
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.customer_mktsegment
ORDER BY avg_discount_pct DESC

**Interpretation:** Reveals which customer segments receive the deepest discounts, showing whether pricing discipline is being maintained.

**Management Action:** Review discount policies for high-discount segments to protect margins; consider volume-based discount tiers aligned with actual purchase behavior.

In [0]:
%sql
-- Q16: Average Ship-to-Receipt Days by Shipping Mode
SELECT
  fs.ship_mode,
  ROUND(AVG(DATEDIFF(fs.receipt_date, fs.ship_date)), 1) AS avg_ship_to_receipt_days,
  ROUND(AVG(DATEDIFF(fs.commit_date, fs.ship_date)), 1) AS avg_ship_to_commit_days,
  COUNT(*) AS line_count
FROM tpch_dev.gold.fact_sales fs
WHERE fs.ship_date IS NOT NULL AND fs.receipt_date IS NOT NULL
GROUP BY fs.ship_mode
ORDER BY avg_ship_to_receipt_days DESC

**Interpretation:** Compares delivery speed across shipping modes, highlighting the most and least efficient logistics options.

**Management Action:** Promote faster shipping modes for time-sensitive orders and negotiate better SLAs with carriers for underperforming modes.

In [0]:
%sql
-- Q17: Order Count and Revenue by Order Priority
SELECT
  fs.order_priority,
  COUNT(DISTINCT fs.order_key) AS order_count,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  ROUND(SUM(fs.net_revenue) / COUNT(DISTINCT fs.order_key), 2) AS avg_revenue_per_order
FROM tpch_dev.gold.fact_sales fs
GROUP BY fs.order_priority
ORDER BY net_revenue DESC

**Interpretation:** Shows how order priority levels are distributed and their revenue impact, indicating whether priority handling aligns with revenue importance.

**Management Action:** Review priority assignment rules to ensure high-value orders receive appropriate handling; rebalance priority criteria if low-priority orders carry disproportionate revenue.

In [0]:
%sql
-- Q18: Monthly Breakdown of Order Status
SELECT
  DATE_TRUNC('month', order_date) AS order_month,
  order_status,
  COUNT(DISTINCT order_key) AS order_count,
  ROUND(SUM(net_revenue), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales
GROUP BY DATE_TRUNC('month', order_date), order_status
ORDER BY order_month, order_status

**Interpretation:** Tracks the monthly mix of order statuses (e.g., pending, shipped, delivered), showing operational pipeline health over time.

**Management Action:** If pending or backlogged orders accumulate in specific months, adjust production scheduling and staffing to clear bottlenecks.

In [0]:
%sql
-- Q19: Product Revenue Ranking Within Each Market Segment
SELECT
  dc.customer_mktsegment AS market_segment,
  dp.product_key,
  dp.product_name,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  ROW_NUMBER() OVER (PARTITION BY dc.customer_mktsegment ORDER BY SUM(fs.net_revenue) DESC) AS rank_in_segment
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
JOIN tpch_dev.gold.dim_product dp ON fs.product_key = dp.product_key
GROUP BY dc.customer_mktsegment, dp.product_key, dp.product_name
QUALIFY rank_in_segment <= 5
ORDER BY market_segment, rank_in_segment

**Interpretation:** Reveals the top products within each customer segment, showing segment-specific product preferences.

**Management Action:** Create segment-specific product bundles and marketing campaigns leveraging these rankings to maximize cross-sell opportunities.

In [0]:
%sql
-- Q20: Revenue Comparison: Customer Geography vs Supplier Geography by Region
SELECT
  fs.customer_region_name AS customer_region,
  fs.supplier_region_name AS supplier_region,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  COUNT(*) AS line_count
FROM tpch_dev.gold.fact_sales fs
GROUP BY fs.customer_region_name, fs.supplier_region_name
ORDER BY net_revenue DESC

**Interpretation:** Maps revenue flows between customer regions and supplier regions, exposing supply chain geography patterns and cross-region dependencies.

**Management Action:** Evaluate whether regional sourcing can be localized to reduce lead times and transportation costs for high-volume customer-supplier region pairs.